<a href="https://colab.research.google.com/github/okayhamamci/FlyRankAI_ML_Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** One row = One unique content item for a specific client on a specific day (Grain: `report_date`, `client_hash_id`, `content_hash_id`).
**Table to use:** `fact_content_daily_performance` (from the Hugging Face warehouse).
**Time window:** A mid-panel month, specifically March 2026 (`2026-03`).
**Predict/Rank (Label or proxy):** We will rank pages based on future clicks/impressions opportunity.
**Deliberately excluded:** Any internal FlyRank product flags (like `priority_score` or `action_type`). They are rule-based decisions, not observable signals, so using them is cheating.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

*   **Feature:** `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `sessions`, `scroll_events`. (These are safe and knowable *before* the decision moment).
*   **Label / Proxy:** Future decline or clicks in the following month (April 2026).
*   **Context:** `client_hash_id`, `content_hash_id`, `report_date`. (Used for grouping/joining, never fed into the model).
*   **Excluded:** `health_score` (because it's the answer key in disguise) and raw URLs/client names (to protect privacy).


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [1]:
import pandas as pd
import duckdb
from google.colab import userdata
from sklearn.tree import DecisionTreeClassifier

print("Connecting DuckDB to Hugging Face...")
# 1. Grab your secret token
hf_token = userdata.get('HF_TOKEN')

# 2. Setup DuckDB with your Hugging Face token
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

# 3. Pull the March 2026 data into a Pandas DataFrame
print("Downloading March 2026 data (this may take a minute)...")
path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
df = con.sql(f"SELECT * FROM read_parquet('{path}')").df()

print("Data loaded successfully!")

# 1. Fact: The Grain Check (Should return 0 if grain is unique)
grain_check = df.groupby(["report_date", "client_hash_id", "content_hash_id"]).size()
duplicates = (grain_check > 1).sum()
print(f"Fact 1 - Grain Check (duplicates): {duplicates} (Expected 0)")

# 2. Fact: Slice's row count and date span
print(f"Fact 2 - Row count: {len(df):,}")
print(f"Fact 2 - Date span: {df['report_date'].min()} to {df['report_date'].max()}")

# 3. Fact: Availability (Filter with IS TRUE)
ga4_available = df[df['ga4_data_available'] == True]
print(f"Fact 3 - Rows with GA4 data available: {len(ga4_available):,} out of {len(df):,}")

print("\n--- Building 5-Feature Frame ---")
# 1. gsc_impressions (Knowable at decision moment: recorded historically by Google)
# 2. gsc_clicks (Knowable at decision moment: recorded historically by Google)
# 3. gsc_avg_position (Knowable at decision moment: recorded historically by Google)
# 4. ga4_data_available (Knowable at decision moment: tracks if client has Analytics connected)
# 5. has_impressions (Knowable at decision moment: simple flag if page had any visibility)

features_df = df[["content_hash_id", "gsc_impressions", "gsc_clicks", "gsc_avg_position", "ga4_data_available"]].copy()
features_df["gsc_impressions"] = features_df["gsc_impressions"].fillna(0)
features_df["gsc_clicks"] = features_df["gsc_clicks"].fillna(0)
features_df["gsc_avg_position"] = features_df["gsc_avg_position"].fillna(0)
features_df["ga4_data_available"] = features_df["ga4_data_available"].fillna(False) # Fill with False instead of 0

# Add our 5th feature
features_df["has_impressions"] = (features_df["gsc_impressions"] > 0).astype(int)

print("\n--- The Trap (Leakage Lesson) ---")

# To run instantly, we just test on a random sample of 50,000 rows
sample_df = features_df.sample(50000, random_state=42)

# Target: Did the page get more than 5 impressions today? (Much more balanced!)
y_sample = (sample_df["gsc_impressions"] > 5).astype(int)

# 1. THE HONEST SCORE
# We only give the model 'avg_position'. It's a weak signal, so it will struggle.
X_honest = sample_df[["gsc_avg_position"]]

model = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_honest, y_sample)
honest_score = model.score(X_honest, y_sample)
print(f"Honest Model Accuracy: {honest_score:.3f}")

# 2. THE TRAP (Leakage)
# We accidentally add the exact number of impressions as a feature!
# By feeding it to the model, it secretly knows the answer immediately.
X_trap = X_honest.copy()
X_trap["TRAP_LEAKED_impressions"] = sample_df["gsc_impressions"]

model_trap = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_trap, y_sample)
trap_score = model_trap.score(X_trap, y_sample)
print(f"Trap Model Accuracy (with leak): {trap_score:.3f} (Suspiciously perfect!)")

# 3. Clean up
print(f"Trap removed! Honest features remaining: {features_df.columns.tolist()}")


Connecting DuckDB to Hugging Face...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Data loaded successfully!
Fact 1 - Grain Check (duplicates): 0 (Expected 0)
Fact 2 - Row count: 9,841,378
Fact 2 - Date span: 2026-03-01 00:00:00 to 2026-03-31 00:00:00
Fact 3 - Rows with GA4 data available: 413,966 out of 9,841,378

--- Building 5-Feature Frame ---

--- The Trap (Leakage Lesson) ---
Honest Model Accuracy: 0.912
Trap Model Accuracy (with leak): 1.000 (Suspiciously perfect!)
Trap removed! Honest features remaining: ['content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_data_available', 'has_impressions']


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**What this data can never tell me:**
This is an unbalanced panel. Many clients have different history depths (e.g., they connected Google Analytics late). The rows before their GA4 start date are zero-filled with `ga4_data_available = False`. If we do not filter carefully using that flag, we will mistakenly assume "no tracking" means "no traffic." Additionally, we can never use this data to claim causal proof (e.g., "Google's algorithm caused this drop"), because this is purely observational data, not an isolated experiment.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.